In [1]:
"""RFI Analysis Module

This script loads observational data, filters a frequency band,
and computes RFI probability over given dimension.

Features
--------
- Command-line interface using argparse
- Optional YAML configuration file
- Structured logging
- Input validation
"""

import os
import xarray as xr
import argparse
import logging
from typing import Optional, Dict
from pathlib import Path
import re

import yaml

In [6]:
# ----------------------------- #
# Logging Configuration
# ----------------------------- #
def setup_logging(level: str = "INFO") -> None:
    """
    Configure logging format and level.

    Parameters
    ----------
    level : str
        Logging level (DEBUG, INFO, WARNING, ERROR)
    """
    logging.basicConfig(
        level=getattr(logging, level.upper(), logging.INFO),
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler("time_prob.log", mode="w"),
            logging.StreamHandler()
        ]
    )


In [7]:
# ----------------------------- #
# YAML Configuration Loader
# ----------------------------- #
def load_config(config_path: Optional[str]) -> Dict:
    """
    Load parameters from a YAML configuration file.

    Parameters
    ----------
    config_path : str or None
        Path to YAML file

    Returns
    -------
    dict
        Configuration dictionary
    """
    if config_path is None:
        return {}

    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r") as file:
        config = yaml.safe_load(file)

    logging.info(f"Loaded configuration from {config_path}")
    return config


In [8]:

# ----------------------------- #
# Input Validation Functions
# ----------------------------- #
def validate_directory(path: str) -> str:
    """
    Validate that a path exists and is a directory.

    Parameters
    ----------
    path : str

    Returns
    -------
    str
        Validated directory path
    """
    path = os.path.expanduser(path)

    if not path:
        raise ValueError("Path cannot be empty.")

    if not os.path.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")

    if not os.path.isdir(path):
        raise NotADirectoryError(f"Not a directory: {path}")

    return path


In [9]:

def validate_frequency(value: float, name: str) -> float:
    """
    Validate frequency input.

    Parameters
    ----------
    value : float
    name : str

    Returns
    -------
    float
    """
    if value is None:
        raise ValueError(f"{name} cannot be None.")

    if value <= 0:
        raise ValueError(f"{name} must be positive.")

    return value



In [ ]:
# ----------------------------- #
# Core RFI Function
# ----------------------------- #
def rfi(data, freq_min: float, freq_max: float, dimension: list) -> xarray.dataArray:
    """
    Extract RFI probability of RFI from the master an counter array in a given dimension.

    Parameters
    ----------
    data : xarray.Datasets
        Must contain counter and master in 5D (time, frequency, baseline, azimuth, elevation) 
    freq_min : float
        Lower frequency bound (MHz)
    freq_max : float
        Upper frequency bound (MHz)
    dimension : list
        The axis to get the probability

    Returns
    -------
    xarray.dataArray
        RFI probability as a function of given dimension
    """

    logging.info("Starting RFI computation")

    # Load xarray.Datasets
    counter = data["counter"]
    master = data["master"]

    # Validate frequency range
    min_value = float(data.frequency[0] * 1e-6)
    max_value = float(data.frequency[-1] * 1e-6)

    if not (freq_min >= min_value and freq_max <= max_value):
        raise ValueError(
            f"Frequency range must be within [{min_value}, {max_value}] MHz"
        )
    
    # Select frequency to compute
    sel_counter = counter.sel(frequency=slice(freq_min * 1e6, freq_max * 1e6))
    sel_master = master.sel(frequency=slice(freq_min * 1e6, freq_max * 1e6))

    # Accessing probability
    axis = ("time", "frequency", "baseline", "azimuth", "elevation")
    new_axis = [ x for x in axis if x not in dimension]

    sum_counter = sel_counter.sum(dim=new_axis)
    sum_master = sel_master.sum(dim=new_axis)

    probability = sum_master / (sum_master + sum_counter)
    logging.debug(f"Extracting probability in {dimension}")

    return probability


In [12]:
# ----------------------------- #
# Argument Parser
# ----------------------------- #
def parse_args():
    """
    Parse command-line arguments.

    Returns
    -------
    argparse.Namespace
    """
    parser = argparse.ArgumentParser(description="RFI Analysis Tool")

    parser.add_argument("--path", type=str, help="Path to data directory")
    parser.add_argument("--freq-min", type=float, help="Minimum frequency (MHz)")
    parser.add_argument("--freq-max", type=float, help="Maximum frequency (MHz)")
    parser.add_argument("--dim", type=list, help="Dimension of the probability to compute")
    parser.add_argument("--output", type=str, help="Path to output zarr file")
    parser.add_argument("--config", type=str, help="Path to YAML config file")
    parser.add_argument("--log-level", type=str, default="INFO",
                        help="Logging level")


    return parser.parse_args()


In [ ]:
# ----------------------------- #
# Main Function
# ----------------------------- #
def main():
    """
    Main execution function.
    """
    args = parse_args()
    setup_logging(args.log_level)

    # Load YAML config if provided
    config = load_config(args.config)

    # Merge CLI + YAML (CLI overrides YAML)
    output = args.output or config.get("output")
    path = args.path or config.get("path")
    freq_min = args.freq_min or config.get("freq_min")
    freq_max = args.freq_max or config.get("freq_max")
    dimension = args.dim or config.get("dim", "time")


    # Validate inputs

    freq_min = validate_frequency(freq_min, "Minimum frequency")
    freq_max = validate_frequency(freq_max, "Maximum frequency")

    logging.info(f"Frequency range: {freq_min}–{freq_max} MHz")
    
    path = validate_directory(path)

    logging.info(f"Using data path: {path}")

    initial = "_".join(x[:4] for x in dimension)

    zarr_file = [
        item for item in path.iterdir()
        if item.is_dir() and item.suffix == ".zarr"
        ]
    
    for file in zarr_file:
        full_path = path + file + "/arr"
        try:
            logging.info(f"Processing {full_path}")
            
            data = xr.open_zarr(full_path, consolidated=False)
            probability = rfi(data, freq_min, freq_max)

            # Saving dataset into zarr file
            ds = probability.to_dataset(name= "probability")
            result_file_name = f"{file}_prob_{initial}.zarr"
            root = Path(__file__).resolve().parent.parent
            result_dirr = os.path.join(root / "results" / "probability" / initial / result_file_name)
            ds.to_zarr(result_dirr, mode="w")

        except Exception as e:

            logging.info(f"Failed: {full_path}")
            logging.exception(e)
 


if __name__ == "__main__":
    '''
    python script.py --path ~/data --freq-min 100 --freq-max 200

    python script.py --config config.yaml
    
    '''
    main()